# icetl — reading a real Iceberg table

A working tour of the framework **as it stands at the end of Phase 3**, run against the
live REST catalog rather than fixtures. Every cell here was executed against
`nyc.yellow_tripdata` (19 columns, 62 files, 41M rows) before being written down.

| Phase | State | What you can exercise below |
|---|---|---|
| 0–1 | done | `session.table()`, `session.sql()`, select / filter / limit, actions, `explain()` |
| 2 | done | predicate + projection pushdown, file pruning, the scan report |
| 3 | done | 273 `F.*` functions, reference conformance on both surfaces |
| 4 | **next** | joins, `groupBy().agg()`, set ops — *DataFrame side not built yet* |
| 12 / 13 / 14 | deferred | merge-on-read reads / writes, decimal promotion |

**Prerequisites.** The REST catalog on `localhost:8182` and MinIO on `localhost:9100`
must be up, and `.env` must point at them. Install the kernel with:

```bash
uv sync --extra dev --extra notebook
uv run python -m ipykernel install --user --name icetl --display-name icetl
```

## 1 · Connect

One wrinkle worth knowing: `resolve_settings()` looks for `.env` in the **current working
directory** and does not walk up. A notebook run from `notebooks/` would therefore find
the catalog URI defaults but no S3 credentials, and fail with `ACCESS_DENIED` on the first
read of table metadata. So we locate the repo root and pass `dotenv_path` explicitly —
which is exactly what that parameter is for.

In [1]:
from __future__ import annotations

import dataclasses
import time
from pathlib import Path

import icetl.sql.functions as F
from icetl.catalog import CatalogRegistry
from icetl.conf import resolve_settings
from icetl.sql import Session


def find_repo_root(start: Path | None = None) -> Path:
    """Nearest ancestor holding a pyproject.toml, so this runs from any directory."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return here


ROOT = find_repo_root()
ENV = ROOT / ".env"
print("repo root:", ROOT)
print(".env     :", ENV, "·", "found" if ENV.is_file() else "MISSING")

repo root: D:\workspace\Transformation_Engine_08172026
.env     : D:\workspace\Transformation_Engine_08172026\.env · found


In [2]:
settings = resolve_settings(dotenv_path=ENV)
session = Session(settings=settings)

print("catalog  :", settings.catalog.uri)
print("s3       :", settings.s3.endpoint)
print("namespace:", settings.default_namespace)
print("ansi     :", settings.sql.ansi_mode)

catalog  : http://localhost:8182
s3       : http://localhost:9100
namespace: ('nyc',)
ansi     : False


### What is actually in the catalog

`resolve_settings()` layers `.env`, the environment and any `icetl.*` keys set on the
builder. The registry below is the same one the session uses, so if this cell lists your
tables the rest of the notebook will find them.

In [3]:
registry = CatalogRegistry(settings)
catalog = registry.get(registry.default_name)

print("catalogs:", registry.names(), "· default:", registry.default_name)
for ns in catalog.list_namespaces():
    for ident in catalog.list_tables(ns):
        print("  ", ".".join(ident))

catalogs: ['rest'] · default: rest
   amazon.deforestation
   nyc.wide_smoke
   nyc.yellow_tripdata
   nyc.yellow_tripdata_wide


## 2 · Schema, without reading any data

Analysis is eager and runs against zero-row Arrow views, so `printSchema()` costs a
catalog round-trip and nothing else — no data files are opened.

In [4]:
TABLE = "nyc.yellow_tripdata"

df = session.table(TABLE)
df.printSchema()
print("columns:", len(df.columns))

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
columns: 19


In [5]:
# The Iceberg side: the partition spec is what makes the pruning in section 4 possible.
tbl = catalog.load_table(tuple(TABLE.split(".")))
print(tbl.spec())
print("data files:", len(list(tbl.scan().plan_files())))

[
  1000: pickup_month: month(2)
]
data files: 62


## 3 · Read some rows

In [6]:
df.select("VendorID", "tpep_pickup_datetime", "trip_distance", "total_amount").limit(5).show()

+--------+--------------------+-------------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|total_amount|
+--------+--------------------+-------------+------------+
|       2| 2024-12-01 00:12:27|         9.76|       51.97|
|       2| 2024-12-01 00:50:35|        20.07|       82.69|
|       2| 2024-12-01 00:18:16|         2.34|       24.72|
|       2| 2024-12-01 00:56:13|         5.05|        36.8|
|       1| 2024-12-01 00:21:17|          4.3|        30.6|
+--------+--------------------+-------------+------------+


## 4 · Pushdown — the point of Phase 2

The filter below is written the way anyone would write it: a **bare date string against a
`timestamp` column**. That exact form used to make PyIceberg's literal binding raise from
inside `plan_files()`; it is now widened to `T00:00:00` so that it prunes.

Watch the `== Scans ==` block at the bottom of `explain()`.

In [7]:
june = (
    session.table(TABLE)
    .filter(F.col("tpep_pickup_datetime") >= "2024-06-01")
    .filter(F.col("tpep_pickup_datetime") < "2024-07-01")
    .select("VendorID", "trip_distance", "total_amount")
)

june.explain(True)

== Logical Plan (icetl) ==
SELECT
  "VendorID",
  "trip_distance",
  "total_amount"
FROM "nyc"."yellow_tripdata"
WHERE
  tpep_pickup_datetime >= '2024-06-01' AND tpep_pickup_datetime < '2024-07-01'

== Optimized Plan ==
  rules: qualify, pushdown_projections, normalize, pushdown_predicates, merge_subqueries, simplify
SELECT
  "yellow_tripdata"."vendorid" AS "VendorID",
  "yellow_tripdata"."trip_distance" AS "trip_distance",
  "yellow_tripdata"."total_amount" AS "total_amount"
FROM "nyc"."yellow_tripdata" AS "yellow_tripdata"
WHERE
  "yellow_tripdata"."tpep_pickup_datetime" < '2024-07-01'
  AND "yellow_tripdata"."tpep_pickup_datetime" >= '2024-06-01'

== Analysed Schema ==
root
 |-- VendorID: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- total_amount: double (nullable = true)

== Physical Plan (DuckDB SQL) ==
SELECT "yellow_tripdata"."vendorid" AS "VendorID", "yellow_tripdata"."trip_distance" AS "trip_distance", "yellow_tripdata"."total_amount" AS "total_am

### Reading the scan report programmatically

`explain()` prints it; `_compile` is where it comes from. The helper below is used for the
A/B that follows. Note that it does **not** execute anything — planning a scan is a
metadata operation.

In [8]:
def scan_stats(frame):
    """Files / columns / bytes this frame would read, without running it."""
    compiled = frame._session._compile(frame._plan, frame._sources, frame.columns)
    return [
        {
            "files": f"{s.files_scanned} / {s.files_total}",
            "columns": f"{len(s.columns)} / {s.total_columns}",
            "MB": round(s.bytes_scanned / 1e6, 1),
            "pushed": s.pushed_filter,
        }
        for s in compiled.scans
    ]


for row in scan_stats(june):
    print(row)

{'files': '3 / 62', 'columns': '4 / 19', 'MB': 61.6, 'pushed': "(tpep_pickup_datetime >= '2024-06-01T00:00:00' AND tpep_pickup_datetime < '2024-07-01T00:00:00')"}


### A/B: the same answer, one eleventh of the I/O

Both queries below count exactly the same rows. The first compares the partition column
directly, so PyIceberg can prune on it. The second wraps that column in `date_trunc`,
which cannot be translated into an Iceberg predicate — so it is left for DuckDB to apply
and every file is read.

This is the honest demonstration: **identical results, very different data volume.**

In [9]:
PRUNES = """
    SELECT count(1) AS trips
    FROM nyc.yellow_tripdata
    WHERE tpep_pickup_datetime >= '2024-06-01'
      AND tpep_pickup_datetime <  '2024-07-01'
"""

NO_PRUNE = """
    SELECT count(1) AS trips
    FROM nyc.yellow_tripdata
    WHERE date_trunc('month', tpep_pickup_datetime) = '2024-06-01'
"""

for label, query in (("pruned", PRUNES), ("not pruned", NO_PRUNE)):
    frame = session.sql(query)
    stats = scan_stats(frame)[0]
    started = time.perf_counter()
    answer = frame.collect()[0]["trips"]
    elapsed = time.perf_counter() - started
    print(
        f"{label:>10}: {answer:>9,} trips · {stats['files']:>7} files · "
        f"{stats['MB']:>6} MB · {elapsed:5.2f}s"
    )

    pruned: 3,539,170 trips ·  3 / 62 files ·   61.6 MB ·  0.33s
not pruned: 3,539,170 trips · 62 / 62 files ·  712.1 MB ·  0.42s


Timings here are modest because MinIO is local and the OS page cache is warm. The column
that matters is **MB** — that is the number that becomes wall-clock time against real
object storage.

## 5 · The wide table — projection pushdown, and what a count costs

Two wide tables exist. `nyc.wide_smoke` is the 1-file fixture; `nyc.yellow_tripdata_wide`
is the real thing — 219 columns, 357 files, 33.6 GB, the same 41M rows as the narrow
table. Sizes first, so the timings below have something to stand on.

In [10]:
for name in ("yellow_tripdata", "wide_smoke", "yellow_tripdata_wide"):
    t = catalog.load_table(("nyc", name))
    files = list(t.scan().plan_files())
    rows = sum(f.file.record_count for f in files)
    gb = sum(f.file.file_size_in_bytes for f in files) / 1e9
    print(
        f"{name:<22} {len(t.schema().fields):>4} cols  {len(files):>4} files  "
        f"{rows:>12,} rows  {gb:>7.1f} GB"
    )

yellow_tripdata          19 cols    62 files    41,169,720 rows      0.7 GB
wide_smoke              219 cols     1 files        50,000 rows      0.0 GB
yellow_tripdata_wide    219 cols   357 files    41,169,720 rows     33.6 GB


### Projection pushdown

Asking for two of 219 columns should read two — this is the case that
`ORDER BY <output alias>` used to silently defeat, sending it back to 219 of 219.

In [11]:
WIDE = "nyc.yellow_tripdata_wide"

wide = session.table(WIDE)
print("columns:", len(wide.columns))
print(scan_stats(wide.select(*wide.columns[:2]))[0])

columns: 219
{'files': '357 / 357', 'columns': '2 / 219', 'MB': 33573.9, 'pushed': None}


### How long does a total count take?

41,169,720 rows across 357 files. Run it a few times — the first is cold, the rest warm.

In [12]:
import statistics

timings = []
for _ in range(5):
    started = time.perf_counter()
    total = session.table(WIDE).count()
    timings.append(time.perf_counter() - started)

print(f"rows: {total:,}")
print("runs:", "  ".join(f"{t:.2f}s" for t in timings))
print(
    f"cold {timings[0]:.2f}s · warm median {statistics.median(timings[1:]):.2f}s "
    f"· best {min(timings):.2f}s"
)

rows: 41,169,720
runs: 5.49s  4.12s  4.48s  3.93s  3.37s
cold 5.49s · warm median 4.03s · best 3.37s


### Where that time goes

Roughly half of it is **Iceberg planning**, not query execution — and that half scales
with file count, not row count.

In [13]:
def phase(label, thunk):
    started = time.perf_counter()
    out = thunk()
    elapsed = time.perf_counter() - started
    print(f"  {label:<40} {elapsed:6.2f}s")
    return out


fresh = session.table(WIDE)
phase("resolve table + bind schema", lambda: fresh.columns)
phase(
    "compile: optimize + plan 357 files",
    lambda: fresh._session._compile(fresh._plan, fresh._sources, fresh.columns),
)
phase("full count (plan + execute)", lambda: session.table(WIDE).count())

print()
phase(
    "same count on the narrow 19-col table",
    lambda: session.sql("SELECT count(1) AS c FROM nyc.yellow_tripdata").collect()[0]["c"],
)

  resolve table + bind schema                0.03s
  compile: optimize + plan 357 files         1.70s
  full count (plan + execute)                3.50s

  same count on the narrow 19-col table      0.29s


41169720

Identical row count, ~8× faster on the narrow table. The difference is 62 files against
357 — per-file planning and footer reads, not rows.

### The scan report's byte figure is not bytes transferred ⚠️

`bytes_scanned` is the **size of the selected files**, and it does not account for column
pruning or for footer-only reads. All three queries below report the same 33.6 GB, but
their times track the number of columns — and a genuine 33.6 GB read in ~2s would need
about 17 GB/s.

In [14]:
some = wide.columns[10:20]
probes = [
    ("count(1)", f"SELECT count(1) AS v FROM {WIDE}"),
    ("sum of 1 column", f"SELECT sum(`{some[0]}`) AS v FROM {WIDE}"),
    (
        "sum of 10 columns",
        "SELECT " + " + ".join(f"sum(`{c}`)" for c in some) + f" AS v FROM {WIDE}",
    ),
]

for label, sql in probes:
    frame = session.sql(sql)
    stats = scan_stats(frame)[0]
    started = time.perf_counter()
    frame.collect()
    elapsed = time.perf_counter() - started
    print(
        f"  {label:<20} {elapsed:5.2f}s   "
        f"plan says {stats['columns']:>7} cols · {stats['MB'] / 1000:.1f} GB"
    )

  count(1)              3.44s   plan says 1 / 219 cols · 33.6 GB
  sum of 1 column       6.87s   plan says 1 / 219 cols · 33.6 GB
  sum of 10 columns    10.59s   plan says 10 / 219 cols · 33.6 GB


### A filtered count is much cheaper — and an unfiltered one need not read data at all

One month prunes to 29 of 357 files. And for a count with no filter, Iceberg already
stores the answer: summing `record_count` across the manifests gives the identical
number without opening a single parquet file.

That last one is **not** what `df.count()` does today — it is an optimisation that is not
implemented, and it belongs to Phase 10 (performance & scale) rather than anything
earlier.

In [15]:
one_month = session.sql(
    f"SELECT count(1) AS c FROM {WIDE} "
    "WHERE tpep_pickup_datetime >= '2024-06-01' "
    "  AND tpep_pickup_datetime <  '2024-07-01'"
)
print("pruned to", scan_stats(one_month)[0]["files"], "files")
rows = phase("filtered count (one month)", lambda: one_month.collect()[0]["c"])
print(f"  -> {rows:,} rows")

print()
wide_tbl = catalog.load_table(tuple(WIDE.split(".")))
from_manifests = phase(
    "count from Iceberg manifests only",
    lambda: sum(f.file.record_count for f in wide_tbl.scan().plan_files()),
)
print(f"  -> {from_manifests:,} rows — same answer, no parquet opened")

pruned to 29 / 357 files
  filtered count (one month)                 0.81s
  -> 3,539,170 rows

  count from Iceberg manifests only          1.81s
  -> 41,169,720 rows — same answer, no parquet opened


## 6 · The function library — 273 names

Phase 3's rule was that every function needs a test asserting on a **value**, not on
generated SQL. Around a dozen of the first 169 produced perfectly plausible SQL and the
wrong answer, with nothing raising.

In [16]:
print("public F.* names:", len(F.__all__))

public F.* names: 273


In [17]:
session.table(TABLE).limit(5).select(
    F.col("trip_distance"),
    F.round(F.col("total_amount"), 1).alias("amount"),
    F.upper(F.col("store_and_fwd_flag")).alias("flag"),
    F.year(F.col("tpep_pickup_datetime")).alias("yr"),
    F.month(F.col("tpep_pickup_datetime")).alias("mon"),
    F.dayofweek(F.col("tpep_pickup_datetime")).alias("dow"),
    F.when(F.col("total_amount") > 50, "high").otherwise("normal").alias("band"),
).show()

+-------------+------+----+----+---+---+------+
|trip_distance|amount|flag|  yr|mon|dow|  band|
+-------------+------+----+----+---+---+------+
|         9.76|  52.0|   N|2024| 12|  1|  high|
|        20.07|  82.7|   N|2024| 12|  1|  high|
|         2.34|  24.7|   N|2024| 12|  1|normal|
|         5.05|  36.8|   N|2024| 12|  1|normal|
|          4.3|  30.6|   N|2024| 12|  1|normal|
+-------------+------+----+----+---+---+------+


### The ones that were silently wrong before value-level testing

Each of these generated believable SQL and the wrong answer. They are pinned now — on
the `F.*` surface, which is the one Phase 3 built.

In [18]:
one = session.table(TABLE).limit(1)
monday = F.lit("2024-06-03").cast("date")  # a Monday

one.select(
    F.split(F.lit("a1b2c"), "[0-9]").alias("split_is_regex"),
    F.greatest(F.lit(1), F.lit(None), F.lit(3)).alias("greatest_skips_null"),
    F.concat_ws("-", F.lit("a"), F.lit(None), F.lit("b")).alias("concat_ws_skips_null"),
    F.log(F.lit(2), F.lit(8)).alias("log_base_first"),
    F.rint(F.lit(2.5)).alias("rint_2_5"),
    F.rint(F.lit(3.5)).alias("rint_3_5"),
    F.weekday(monday).alias("monday_is_0"),
    F.split_part(F.lit("a,b"), F.lit(","), F.lit(9)).alias("oob_is_empty"),
).show(truncate=False)

+--------------+-------------------+--------------------+--------------+--------+--------+-----------+------------+
|split_is_regex|greatest_skips_null|concat_ws_skips_null|log_base_first|rint_2_5|rint_3_5|monday_is_0|oob_is_empty|
+--------------+-------------------+--------------------+--------------+--------+--------+-----------+------------+
|[a, b, c]     |3                  |a-b                 |3.0           |2.0     |4.0     |0          |            |
+--------------+-------------------+--------------------+--------------+--------+--------+-----------+------------+


## 7 · the reference conformance

DuckDB's defaults are not the reference's. These rules run as one tree pass *before* the
optimizer, so pushdown sees the tree that actually executes.

In [19]:
session.sql("""
    SELECT 1 / 0              AS int_div,
           1.0 / 0.0          AS decimal_div,
           CAST('abc' AS INT) AS bad_cast,
           NULL <=> NULL      AS null_safe_eq
""").show()

+-------+-----------+--------+------------+
|int_div|decimal_div|bad_cast|null_safe_eq|
+-------+-----------+--------+------------+
|   NULL|       NULL|    NULL|        true|
+-------+-----------+--------+------------+


`1.0 / 0.0` is worth a second look: it used to **crash** with `decimal.DivisionByZero`
raised from inside sqlglot's `simplify` rule, on both surfaces. The integer spelling
`1 / 0` was tested and could never have found it — `simplify` declines to fold integer
division at all.

### NULL ordering — the reference puts nulls first ascending, last descending

In [20]:
rows = "(VALUES (2), (NULL), (1)) AS t(x)"
session.sql(f"SELECT x FROM {rows} ORDER BY x ASC").show()
session.sql(f"SELECT x FROM {rows} ORDER BY x DESC").show()

+----+
|   x|
+----+
|NULL|
|   1|
|   2|
+----+
+----+
|   x|
+----+
|   2|
|   1|
|NULL|
+----+


### ANSI mode

Off by default, as in the reference. Turning it on opts into strict casting only — a failed cast
raises instead of giving NULL. It does *not* make integer overflow wrap, which DuckDB
cannot do.

In [21]:
strict = Session(
    settings=dataclasses.replace(settings, sql=dataclasses.replace(settings.sql, ansi_mode=True))
)

try:
    strict.sql("SELECT CAST('abc' AS INT)").collect()
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc).splitlines()[0]}")
finally:
    strict.stop()

QueryExecutionException: ConversionException: Conversion Error: Could not convert string 'abc' to INT32


## 8 · P1 — both surfaces, one path

`session.sql()` and the DataFrame API differ only in how the plan is *built*. From
substitution down there is a single code path, so the two must agree. Phase 3's
conformance rules are a tree pass rather than methods on `Column` for exactly this
reason: a rule inside `Column` would cover one surface and miss the other, and the two
would quietly disagree.

In [22]:
via_sql = session.sql(PRUNES).collect()[0]["trips"]

via_df = (
    session.table(TABLE)
    .filter(F.col("tpep_pickup_datetime") >= "2024-06-01")
    .filter(F.col("tpep_pickup_datetime") < "2024-07-01")
    .count()
)

print(f"SQL surface       : {via_sql:,}")
print(f"DataFrame surface : {via_df:,}")
assert via_sql == via_df, "P1 violated: the two surfaces disagree"
print("agree ✓")

SQL surface       : 3,539,170
DataFrame surface : 3,539,170
agree ✓


### …but P1 holds for the *plan*, not yet for the function library ⚠️

Relational operators go through one path, and the conformance rules in
`sql/conformance.py` are a tree pass that both surfaces cross. But a function whose the reference
semantics are implemented by **composition inside `functions.py`** — `rint` picking the
even neighbour on a tie, `weekday` shifting Monday to 0 — only exists on the `F.*`
surface. `session.sql("SELECT rint(2.5)")` hands the name straight to DuckDB.

The probe below measures the gap. Two classes come out of it, and they are not equally
dangerous:

- **Missing** — DuckDB has no such function, so the query *raises*. Loud, and safe.
- **Silently different** — DuckDB has a function of the same name with different
  semantics. No error, wrong answer. This is precisely the failure mode Phase 3's
  value-level testing rule exists to catch, and why `date_part('DOW', …)` was made to
  refuse rather than answer.

In [23]:
one_row = session.table(TABLE).limit(1)

# (label, SQL spelling, F.* builder, the reference's documented answer)
CASES = [
    ("rint(2.5)", "rint(2.5)", lambda: F.rint(F.lit(2.5)), 2.0),
    ("weekday(Mon)", "weekday(DATE '2024-06-03')", lambda: F.weekday(monday), 0),
    ("dayofweek(Mon)", "dayofweek(DATE '2024-06-03')", lambda: F.dayofweek(monday), 2),
    ("log1p(0)", "log1p(0)", lambda: F.log1p(F.lit(0)), 0.0),
    (
        "find_in_set",
        "find_in_set('b', 'a,b,c')",
        lambda: F.find_in_set(F.lit("b"), F.lit("a,b,c")),
        2,
    ),
    ("octet_length", "octet_length('h\u00e9llo')", lambda: F.octet_length(F.lit("h\u00e9llo")), 6),
    (
        "next_day",
        "next_day(DATE '2024-06-03', 'Mon')",
        lambda: F.next_day(monday, "Mon"),
        "2024-06-10",
    ),
    (
        "element_at",
        "element_at(array(10,20,30), 1)",
        lambda: F.element_at(F.array(F.lit(10), F.lit(20), F.lit(30)), 1),
        10,
    ),
    ("equal_null", "equal_null(NULL, NULL)", lambda: F.equal_null(F.lit(None), F.lit(None)), True),
    ("size(NULL)", "size(NULL)", lambda: F.size(F.lit(None)), -1),
]


def attempt(thunk):
    try:
        return True, thunk()
    except Exception as exc:
        return False, type(exc).__name__


def as_sql(expr):
    return attempt(lambda: session.sql(f"SELECT {expr} AS v").collect()[0]["v"])


def as_df(build):
    return attempt(lambda: one_row.select(build().alias("v")).collect()[0]["v"])

In [24]:
import datetime


def norm(v):
    return v.isoformat() if isinstance(v, datetime.date) else v


def same(a, b):
    return norm(a) == norm(b)


print(f"{'function':<16}{'SQL surface':<22}{'F.* surface':<16}{'the reference':<12}verdict")
print("-" * 78)

diverged, wrong_on_both = [], []
for label, expr, build, expected in CASES:
    sql_ok, sql_val = as_sql(expr)
    df_ok, df_val = as_df(build)

    if not (sql_ok and df_ok) or not same(sql_val, df_val):
        verdict = "DIVERGE" if sql_ok else "missing in SQL"
        diverged.append(label)
    elif not same(df_val, expected):
        verdict = "agree, but != the reference"
        wrong_on_both.append(label)
    else:
        verdict = "agree"

    print(f"{label:<16}{sql_val!s:<22.21}{df_val!s:<16.15}{expected!s:<12.11}{verdict}")

print("-" * 78)
print(f"{len(diverged)} of {len(CASES)} diverge between surfaces: {', '.join(diverged)}")
print(f"{len(wrong_on_both)} agree but contradict the reference: {', '.join(wrong_on_both) or '-'}")

function        SQL surface           F.* surface     the referenceverdict
------------------------------------------------------------------------------
rint(2.5)       AnalysisException     2.0             2.0         missing in SQL
weekday(Mon)    1                     0               0           DIVERGE
dayofweek(Mon)  1                     2               2           DIVERGE
log1p(0)        AnalysisException     0.0             0.0         missing in SQL
find_in_set     AnalysisException     2               2           missing in SQL
octet_length    AnalysisException     6               6           missing in SQL
next_day        2024-06-10            2024-06-10      2024-06-10  agree
element_at      10                    10              10          agree
equal_null      True                  True            True        agree
size(NULL)      None                  None            -1          agree, but != the reference
----------------------------------------------------------------

**Read the two lines at the bottom carefully.**

`weekday` and `dayofweek` are the ones to worry about: both surfaces answer, neither
raises, and the SQL surface is off by one — DuckDB's numbering, not the reference's. An
off-by-one weekday is invisible in a result set.

`size(NULL)` is a third thing again: both surfaces agree, and both contradict the reference.
`array_size`'s own docstring in `functions.py` says *"the reference's `size` answers -1 there by
default; `array_size` answers NULL. The two exist to differ, so neither can be an alias
of the other"* — but `size` is implemented as `sg.ArraySize`, which yields NULL, making
them aliases after all.

None of this is reachable from the DataFrame API alone, which is where Phase 3's tests
live. It is a **Phase 4 input**, not a blocker for anything above this cell.

## 9 · Aggregation today

`groupBy().agg()` is **Phase 4** and is not on the DataFrame yet. The SQL surface,
however, goes straight through sqlglot to DuckDB — so aggregates already work there,
with the Phase 2 pushdown underneath them.

Note the backticks. In the reference, `"VendorID"` is a *string literal*, not an identifier;
icetl follows the reference, so double quotes would hand you a column of the constant text
`VendorID` rather than the column itself.

In [25]:
revenue = session.sql("""
    SELECT `VendorID`,
           count(1)                     AS trips,
           round(sum(total_amount), 2)  AS revenue,
           round(avg(trip_distance), 3) AS avg_miles
    FROM nyc.yellow_tripdata
    WHERE tpep_pickup_datetime >= '2024-06-01'
      AND tpep_pickup_datetime <  '2024-07-01'
    GROUP BY `VendorID`
    ORDER BY trips DESC
""")

print(scan_stats(revenue)[0])
revenue.show()

{'files': '3 / 62', 'columns': '4 / 19', 'MB': 61.6, 'pushed': "(tpep_pickup_datetime >= '2024-06-01T00:00:00' AND tpep_pickup_datetime < '2024-07-01T00:00:00')"}
+--------+-------+-----------+---------+
|VendorID|  trips|    revenue|avg_miles|
+--------+-------+-----------+---------+
|       2|2700108|76092574.07|    5.501|
|       1| 839052|22761325.93|    4.326|
|       6|     10|     390.75|   11.018|
+--------+-------+-----------+---------+


Two things to notice in that scan line: the aggregate reads **3 of 62 files** and only
the columns it names. `ORDER BY trips` — an output alias, not a table column — is exactly
what used to disable projection pushdown entirely.

## 10 · Out to pandas

In [26]:
pdf = revenue.toPandas()
print(pdf.dtypes)
pdf

VendorID       int32
trips          int64
revenue      float64
avg_miles    float64
dtype: object


,VendorID,trips,revenue,avg_miles
0,2,2700108,76092574.07,5.501
1,1,839052,22761325.93,4.326
2,6,10,390.75,11.018


## 11 · Sharp edges — know these before you trust a result

Everything below is **known and documented** in `src/icetl/compat/divergence.md`. None of
it raises, which is exactly why it is worth seeing once.

### `date_format` patterns are not translated ⚠️

the reference's pattern language is Java's (`yyyy-MM-dd`); DuckDB's is strftime (`%Y-%m-%d`).
They are different languages and icetl does **not** translate between them — so the reference
spelling silently hands the pattern back to you as a literal string. This one has no
value-level test behind it; treat it as the live trap in the current surface.

In [27]:
session.table(TABLE).limit(3).select(
    F.date_format(F.col("tpep_pickup_datetime"), "yyyy-MM").alias("java_spelling"),
    F.date_format(F.col("tpep_pickup_datetime"), "%Y-%m").alias("duckdb_spelling"),
).show()

+-------------+---------------+
|java_spelling|duckdb_spelling|
+-------------+---------------+
|      yyyy-MM|        2024-12|
|      yyyy-MM|        2024-12|
|      yyyy-MM|        2024-12|
+-------------+---------------+


### Decimal division returns DOUBLE (decision 14, deferred to Phase 14)

the reference gives `DECIMAL(16,6)` for `DECIMAL(10,2) / DECIMAL(10,2)`; DuckDB gives `DOUBLE`.
That is accurate to ~15 significant digits — wrong only where exact decimal semantics
were the point, which for a money column is precisely when it matters.

`+` and `-` already agree. `*` keeps the right scale with a smaller precision, overflowing
only at extreme magnitudes.

In [28]:
session.sql("""
    SELECT typeof(a + b) AS add_type,
           typeof(a * b) AS mul_type,
           typeof(a / b) AS div_type
    FROM (SELECT CAST(1.00 AS DECIMAL(10,2)) AS a,
                 CAST(3.00 AS DECIMAL(10,2)) AS b)
""").show()

+-------------+-------------+--------+
|     add_type|     mul_type|div_type|
+-------------+-------------+--------+
|DECIMAL(11,2)|DECIMAL(18,4)|  DOUBLE|
+-------------+-------------+--------+


### Functions that refuse rather than mislead

Two classes of deliberate refusal. `rand(seed)` raises because DuckDB seeds per
*connection*, not per expression — accepting the seed would return unseeded values from
the one argument that exists for reproducibility. `date_part('DOW', …)` raises because
the reference numbers Sunday 1 and DuckDB numbers it 0, and an off-by-one weekday looks perfectly
fine in the output.

In [29]:
probes = [
    ("F.rand(42)", lambda: F.rand(42)),
    ("F.date_part('DOW', c)", lambda: F.date_part("DOW", F.col("tpep_pickup_datetime"))),
    ("Column.over()  → Phase 5", lambda: F.col("x").over(None)),
]

for label, call in probes:
    try:
        call()
        print(f"{label:<28} did not raise")
    except Exception as exc:
        print(f"{label:<28} {type(exc).__name__}: {str(exc).splitlines()[0][:80]}")

F.rand(42)                   EngineValueError: rand(seed=...) is not supported: DuckDB seeds its generator per connection, not 
F.date_part('DOW', c)        EngineValueError: date_part() does not support the field 'DOW'. For the day of week use F.dayofwee
Column.over()  → Phase 5     UnsupportedFeatureError: Column.over() is not implemented yet. It is scheduled for Phase 5.


### Merge-on-read is refused, not silently mis-read

`read_parquet` cannot see an Iceberg delete file, so a merge-on-read table would return
deleted rows and report success. `_assert_copy_on_write` refuses the scan instead, naming
Phase 12 in the error.

The tables in this catalog are all copy-on-write, so the guard does not fire here — the
`fx.mor` fixture in the local suite is what proves that it does. That guard is also what
makes the Phase 12 deferral safe to sit on: if a merge-on-read table ever appears
upstream, queries start *failing* rather than quietly returning deleted rows.

### Float aggregation is not bit-stable

`sum(double)` over parallel scans varies in the last few digits between runs, because
DuckDB's aggregation order varies. Expected for floats — the reference does the same. Run this a
few times; it may print one result or several.

In [30]:
sums = {
    session.sql(
        "SELECT sum(total_amount) AS s FROM nyc.yellow_tripdata "
        "WHERE tpep_pickup_datetime >= '2024-06-01' "
        "  AND tpep_pickup_datetime <  '2024-07-01'"
    ).collect()[0]["s"]
    for _ in range(3)
}

print(f"{len(sums)} distinct result(s) across 3 runs:")
for value in sums:
    print(" ", repr(value))

2 distinct result(s) across 3 runs:
  98854290.75010766
  98854290.75010765


## 12 · Done

In [31]:
session.stop()
print("session stopped")

session stopped


---

## What this run establishes

Working, on a real 41M-row table:

- Predicate pushdown prunes **3 of 62 files** — 61.6 MB against 712.1 MB for the same
  answer.
- Projection pushdown reads **2 of 219 columns** on the wide table.
- A total count of the 219-column, 357-file, 33.6 GB table: **~2s warm, ~3.3s cold**,
  about half of it Iceberg planning. The same 41M rows on the 62-file narrow table count
  in ~0.25s — the cost tracks *files*, not rows.
- Both surfaces agree on the relational plan, and the conformance rules hold on each.
- 273 `F.*` names, with the compositions (`rint`, `weekday`, `overlay`, …) correct.

Found by running it, and **not covered by the current tests**:

| # | What | Severity |
|---|---|---|
| 1 | `weekday` / `dayofweek` are **silently off by one** through `session.sql()` — DuckDB's numbering reaches the answer unchanged | wrong answers, no error |
| 2 | ~10 composed functions (`rint`, `log1p`, `find_in_set`, `octet_length`, `overlay`, `width_bucket`, …) do not exist on the SQL surface at all | raises — loud, safe |
| 3 | `size(NULL)` gives NULL on both surfaces; the reference says `-1`, and `array_size`'s own docstring says the two must differ | wrong on both surfaces |
| 4 | `resolve_settings()` reads `.env` from `Path.cwd()` only, with no upward walk — hence the `dotenv_path` in section 1 | setup friction |
| 5 | `bytes_scanned` in the scan report is the selected files' size, not bytes read — it ignores column pruning, so `explain()` overstates a narrow query's cost | misleading, not wrong |
| 6 | An unfiltered `count(*)` reads parquet footers when Iceberg's manifests already hold the answer — ~2× here, and it grows with file count | missed optimisation |

Items 1–3 are one root cause: Phase 3's conformance work lives in `F.*` composition and
in `sql/conformance.py`, but a bare function name in `session.sql()` goes straight to
DuckDB. Phase 3's tests exercise the `F.*` surface, so none of this is reachable from
them. That makes it a **Phase 4 input** — P1 is stated over both surfaces, and item 1 is
the class of defect the value-level testing rule exists to catch.

Items 5 and 6 are both **Phase 10** (performance & scale), and neither affects
correctness: 5 makes `explain()` overstate what a narrow query costs, and 6 leaves a
cheap win on the table for unfiltered counts.

## What to try next

- Point `TABLE` at your own table — nothing above is specific to the NYC data except the
  column names in sections 3, 6 and 9.
- Compare `scan_stats()` before and after adding a filter on your partition column. If
  `files` does not drop, the predicate did not translate; the `pushed filters:` line in
  `explain()` tells you what did.
- Extend the section 8 probe. It is ten cases against a 273-name surface; the same shape
  run over all of `F.__all__` would size the gap properly.
- Phase 4 opens the DataFrame side of aggregation, joins and set operations. Two
  carry-over notes are waiting for it: set operations get no pushdown today (the
  optimizer declines to rewrite a `UNION` whose output names need restoring), and two
  references to one table merge into a single scan — correct, but a self-join with
  disjoint filters prunes less than it could.